# 02 — Feature engineering (an NVTabular workflow, reimplemented in pandas)

NVIDIA's NVTabular library builds a **`Workflow`**: a declarative chain of
feature-engineering ops (`Categorify`, `Normalize`, groupby/list ops for
sequences, …) that gets *fit* once on training data and then *applied*
identically at both training and serving time. That train/serve consistency
is the actual point of NVTabular — not the ops themselves, which are things
you could write in pandas. I can't install the real NVTabular package in this
local environment (see ADR-007 in `docs/ARCHITECTURE_DECISIONS.md` — it pins
`pandas<1.6`, which no longer resolves cleanly against anything modern on this
machine), so this notebook reimplements the *same* fit/transform split by hand,
in plain pandas, to learn the mechanism. `training/modal_app.py` runs the real
`nvtabular.Workflow` for the production training run, inside NVIDIA's own
container, where none of this version conflict exists.

What this notebook builds, concretely:
1. **Categorify** — map `userId`, `movieId`, and `genres` to contiguous integer
   ids, with a reserved "unknown" index for anything not seen during fitting.
2. **Context derivation** — `day_of_week` and `time_of_day`, from the rating
   timestamp (see notebook 1 for why this is a real, if noisy, context
   signal).
3. **The "user-take" feature** — each rating's *preceding* K ratings by the
   same user, as a sequence of movie ids. This is a groupby + shift + rolling
   window, and it has to respect time order or it leaks the future into the
   features.
4. **A time-based train/val/test split** — leave-one-out per user, not a
   random split, because random splits let the model see a user's "future"
   ratings during training.

No demographics here, by the way — `ml-latest-small` only ships `userId`, no
age/gender/occupation. (The older `ml-1m` release has those, but no tags —
I picked `ml-latest-small` for the real text signal in notebook 3 instead; see
the dataset note in the README.)


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("..") / "data" / "raw" / "ml-latest-small"
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")

ratings["ts"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings = ratings.sort_values(["userId", "ts"]).reset_index(drop=True)
ratings.head(3)


,userId,movieId,rating,timestamp,ts
0,1,804,4.0,964980499,2000-07-30 18:08:19
1,1,1210,5.0,964980499,2000-07-30 18:08:19
2,1,2018,5.0,964980523,2000-07-30 18:08:43


## Step 1 — Categorify: userId and movieId → contiguous integer ids

In [2]:
UNK = 0  # reserved index for "unknown / unseen at serving time"


def fit_vocab(values):
    """The 'fit' half of Categorify: build value -> contiguous-id mapping."""
    uniques = sorted(pd.unique(values))
    return {str(v): i + 1 for i, v in enumerate(uniques)}  # 0 reserved for UNK


def transform_vocab(values, vocab):
    """The 'transform' half: apply a *fitted* mapping, OOV -> UNK."""
    return values.map(lambda v: vocab.get(str(v), UNK)).astype("int64")


user_vocab = fit_vocab(ratings["userId"])
movie_vocab = fit_vocab(ratings["movieId"])

ratings["user_idx"] = transform_vocab(ratings["userId"], user_vocab)
ratings["movie_idx"] = transform_vocab(ratings["movieId"], movie_vocab)

print(f"users (incl. UNK slot): {len(user_vocab) + 1}")
print(f"movies (incl. UNK slot): {len(movie_vocab) + 1}")


users (incl. UNK slot): 611
movies (incl. UNK slot): 9725


The crucial thing about `fit_vocab`/`transform_vocab` being two separate
functions, not one: at serving time, I only ever call `transform_vocab` with
the vocab that was fit once during training. If I refit the vocab on every
request, a user's id could map to a *different* integer each time, and the
model's learned embedding table would point at the wrong row. This is exactly
the train/serve consistency NVTabular's `Workflow.fit`/`.transform` split
exists to guarantee — I'm just doing it by hand here.


## Step 2 — Categorify genres as a multi-valued (list) column

In [3]:
genres_split = movies["genres"].str.split("|")
all_genres = sorted({g for gs in genres_split for g in gs if g != "(no genres listed)"})
genre_vocab = {g: i + 1 for i, g in enumerate(all_genres)}  # 0 = UNK/no-genre

movies["genre_idxs"] = genres_split.map(
    lambda gs: [genre_vocab[g] for g in gs if g in genre_vocab] or [0]
)

print(f"distinct genres: {len(genre_vocab)}")
movies[["movieId", "genres", "genre_idxs"]].head(3)


distinct genres: 19


,movieId,genres,genre_idxs
0,1,Adventure|Animation|Children|Comedy|Fantasy,"[2, 3, 4, 5, 9]"
1,2,Adventure|Children|Fantasy,"[2, 4, 9]"
2,3,Comedy|Romance,"[5, 15]"


This is a *list* feature — each movie has a variable number of genres. NVTabular
has first-class support for this (it calls them "multi-hot" / list columns);
on the model side, the item tower will embed each genre id and **mean-pool**
the list into a single fixed-size vector, so one movie's "3 genres" and
another's "1 genre" both end up as the same-sized input.


## Step 3 — context: day_of_week, time_of_day

In [4]:
def time_of_day(hour: int) -> str:
    if 5 <= hour < 12:
        return "morning"
    if 12 <= hour < 17:
        return "afternoon"
    if 17 <= hour < 22:
        return "evening"
    return "night"


ratings["day_of_week"] = ratings["ts"].dt.day_name()
ratings["time_of_day"] = ratings["ts"].dt.hour.map(time_of_day)

day_vocab = fit_vocab(ratings["day_of_week"])
tod_vocab = fit_vocab(ratings["time_of_day"])

ratings["day_idx"] = transform_vocab(ratings["day_of_week"], day_vocab)
ratings["tod_idx"] = transform_vocab(ratings["time_of_day"], tod_vocab)

ratings[["day_of_week", "day_idx", "time_of_day", "tod_idx"]].drop_duplicates().sort_values("day_idx")


,day_of_week,day_idx,time_of_day,tod_idx
6476,Friday,1,afternoon,1
261,Friday,1,night,4
443,Friday,1,morning,3
1119,Friday,1,evening,2
300,Monday,2,morning,3
1265,Monday,2,afternoon,1
446,Monday,2,evening,2
994,Monday,2,night,4
232,Saturday,3,evening,2
1256,Saturday,3,night,4


**Note on "device":** MovieLens has no device field at all, so there's no
`device_idx` column here. The frontend will still show a device toggle, purely
to illustrate *where* real context would plug into this same architecture —
clearly labeled as non-functional, not quietly faked.


## Step 4 — the \"user-take\" feature: each user's preceding K ratings

In [5]:
K = 5


def build_user_take(df: pd.DataFrame, k: int) -> pd.Series:
    """For each row, the movie_idx of up to k ratings *strictly before* it,
    by the same user, most-recent-first. Padded with 0 (UNK/empty slot)."""
    out = []
    for _, group in df.groupby("userId", sort=False):
        movie_idxs = group["movie_idx"].tolist()
        for i in range(len(movie_idxs)):
            history = movie_idxs[max(0, i - k):i][::-1]  # preceding k, most recent first
            padded = history + [0] * (k - len(history))
            out.append(padded)
    return pd.Series(out, index=df.index)


ratings["user_take"] = build_user_take(ratings, K)
ratings[["userId", "movieId", "ts", "user_take"]].head(8)


,userId,movieId,ts,user_take
0,1,804,2000-07-30 18:08:19,"[0, 0, 0, 0, 0]"
1,1,1210,2000-07-30 18:08:19,"[633, 0, 0, 0, 0]"
2,1,2018,2000-07-30 18:08:43,"[911, 633, 0, 0, 0]"
3,1,2628,2000-07-30 18:08:43,"[1493, 911, 633, 0, 0]"
4,1,2826,2000-07-30 18:08:43,"[1979, 1493, 911, 633, 0]"
5,1,3578,2000-07-30 18:11:08,"[2126, 1979, 1493, 911, 633]"
6,1,3617,2000-07-30 18:11:23,"[2671, 2126, 1979, 1493, 911]"
7,1,3744,2000-07-30 18:11:34,"[2693, 2671, 2126, 1979, 1493]"


Look at the first row for any given user: its `user_take` is `[0, 0, 0, 0,
0]` — no history yet. That's the natural cold-start case for *this* user,
right here in the real data, not simulated. Over a user's rating history, the
list fills up with real preceding movie ids. This feature is the thing that
lets the model make a reasonable recommendation even when it has learned
nothing about *this specific user id* yet.


## Step 5 — the cold-start training trick: drop the user id for some rows

In [6]:
rng = np.random.default_rng(42)
COLD_START_DROP_FRACTION = 0.15

drop_mask = rng.random(len(ratings)) < COLD_START_DROP_FRACTION
ratings["user_idx_for_training"] = ratings["user_idx"].where(~drop_mask, UNK)

print(f"rows with user id intentionally dropped for training: {drop_mask.sum()} "
      f"({drop_mask.mean():.1%})")


rows with user id intentionally dropped for training: 15031 (14.9%)


For ~15% of training rows, the model never sees the real `user_idx` — only
`UNK` + that row's `user_take` + context. That forces part of the training
signal to flow through "can I recommend well using *only* recent-interaction
history and context, with no learned user embedding at all?" — which is
exactly the situation a brand-new user is in. Notebook 4 evaluates this
directly: Recall@K on a cold-start-simulated slice vs. the full model.


## Step 6 — time-based split: leave-one-out per user

In [7]:
def leave_one_out_split(df: pd.DataFrame):
    df = df.sort_values(["userId", "ts"])
    rank_from_end = df.groupby("userId").cumcount(ascending=False)
    test = df[rank_from_end == 0]
    val = df[rank_from_end == 1]
    train = df[rank_from_end >= 2]
    return train, val, test


train_df, val_df, test_df = leave_one_out_split(ratings)
print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")
assert len(val_df) == len(test_df) == ratings["userId"].nunique()


train: 99616  val: 610  test: 610


Every user contributes exactly one row to val and one to test (their two most
recent ratings), and everything earlier goes to train. A **random** split
would let the model train on a rating that happened chronologically after one
of its "held out" ratings — which is a subtle but real evaluation leak for
anything using time-ordered features like `user_take`.


## Save processed artifacts for notebooks 3 & 4

In [8]:
keep_cols = [
    "userId", "movieId", "user_idx", "movie_idx", "user_idx_for_training",
    "day_idx", "tod_idx", "user_take", "ts", "rating",
]

train_df[keep_cols].to_parquet(PROCESSED_DIR / "train.parquet")
val_df[keep_cols].to_parquet(PROCESSED_DIR / "val.parquet")
test_df[keep_cols].to_parquet(PROCESSED_DIR / "test.parquet")
movies[["movieId", "genre_idxs"]].to_parquet(PROCESSED_DIR / "movie_genres.parquet")

vocab = {
    "user_vocab": user_vocab,
    "movie_vocab": movie_vocab,
    "genre_vocab": genre_vocab,
    "day_vocab": day_vocab,
    "tod_vocab": tod_vocab,
    "user_take_k": K,
}
with open(PROCESSED_DIR / "vocab.json", "w") as f:
    json.dump(vocab, f)

print("wrote processed train/val/test parquet + vocab.json to", PROCESSED_DIR.resolve())


wrote processed train/val/test parquet + vocab.json to /Users/ashish/ws/recommendation-system-merlin/data/processed


## My take — ADR-002

> NVTabular workflow vs. a hand-rolled pandas/sklearn pipeline — what did the
> workflow abstraction actually buy, concretely, over just writing this logic
> by hand?

Having now written the fit/transform split by hand: the actual value isn't
the individual ops (Categorify is a dict lookup; the list/multi-hot handling
is `explode`/`groupby` dressed up). It's that NVTabular makes the **fit vs.
transform boundary structural**, not a convention you have to remember to
respect — the `Workflow` object *is* the serialized vocab + stats, and you
cannot accidentally call `fit_transform` in production and get a
silently-wrong mapping. Write your own take in `docs/ARCHITECTURE_DECISIONS.md`
once you've also felt what it's like to wire this hand-rolled version into the
FastAPI backend in a later step — does the discipline hold up without the
library enforcing it?
